
# Semantic Link & Semantic Link Labs



## 1. Setup
`sempy` is pre-installed in Fabric. Upgrade to get the full `semantic-link-labs` toolkit.


In [ ]:

!pip install -U semantic-link-labs


In [ ]:

# Import semantic link and alias
import sempy.fabric as fabric
from sempy.relationships import plot_relationship_metadata

dataset_name = "Contoso10K"



## 2. Explore the Model



### Explore semantic model


In [ ]:

df_datasets = fabric.list_datasets()
df_datasets



### List tables


In [ ]:

df_tables = fabric.list_tables(dataset_name)
df_tables



### Display columns (including hidden properties)


In [ ]:

df_columns = fabric.list_tables(dataset_name, include_columns=True)
df_columns[df_columns["Name"] == "Sales"]



## 3. Visualize Relationships



### Plot relationship diagram


In [ ]:

contoso_model = "Contoso10K"
relationships = fabric.list_relationships(contoso_model)

plot_relationship_metadata(relationships)



### See the invisible — AutoDate hidden relationships


In [ ]:

__autodate_model = "Contoso10K_WithAutoDate"

relationships_date = fabric.list_relationships(__autodate_model)
plot_relationship_metadata(relationships_date)



## 4. Read Data from the Model



### List measures


In [ ]:

df_measures = fabric.list_measures(dataset_name)
df_measures



### Read a table directly


In [ ]:

df_table = fabric.read_table(dataset_name, "Product")
df_table.head(5)



### Query the model with Spark SQL


In [ ]:

spark.conf.set("spark.sql.catalog.pbi", "com.microsoft.azure.synapse.ml.powerbi.PowerBICatalog")


In [ ]:

df = spark.sql("SHOW TABLES FROM pbi")
display(df)



## 5. Evaluate Measures


In [ ]:

fabric.evaluate_measure(dataset_name, measure="Sales Amount")



### GroupBy Columns


In [ ]:

fabric.evaluate_measure(dataset_name, measure="Sales Amount", groupby_columns=["Product[Brand]", "Product[Category]"])



### DAX cell magic (%%dax)


In [ ]:

# Load %%dax cell magic
%load_ext sempy


In [ ]:

%%dax "Contoso10K"

EVALUATE
SUMMARIZECOLUMNS(
"Sales Amount",[Sales Amount]
)



### Run DAX with evaluate_dax


In [ ]:

__dataset = "Contoso10K"
__workspace = "session_data_on_fire_fabric_spark"

__dax = """
EVALUATE 
SUMMARIZECOLUMNS(
    'Product'[Brand],
   'Date'[Month Short],
   'Date'[Year],
   "Sales Amount",
    CALCULATE([Sales Amount])
    )
"""

fabric.evaluate_dax(
    dataset=__dataset,
    dax_string= __dax,
    workspace=__workspace,
    verbose=0,
    num_rows=10
)



## 6. Data Quality — Referential Integrity
Use DAX to surface Sales dates that don't exist in the Date table — the classic blank row problem.


In [ ]:

__dataset = "Contoso10K_RI"
__workspace = "session_data_on_fire_fabric_spark"

dax = """
EVALUATE 
    EXCEPT (
        VALUES( 'Sales'[Order Date] ),
        VALUES('Date'[Date] )
    )
"""

fabric.evaluate_dax(
    dataset=__dataset,
    dax_string= dax,
    workspace=__workspace,
    verbose=0,
    num_rows=None
)



> **Finding:** 2017 is missing from the Date table — either it was never loaded or was filtered out in Power Query.



## 7. Model Health — Memory Analyzer
One of the tools in the Semantic Link Labs model health suite.


In [ ]:

import sempy.fabric as fabric

__smallmodel = "Contoso10K"
__workspace = "session_data_on_fire_fabric_spark"

fabric.model_memory_analyzer(dataset = __smallmodel, workspace = __workspace)



### Compare: model bloated with AutoDate


In [ ]:

import sempy.fabric as fabric

__xlmodel = "Contoso10K_WithAutoDateXL"
__workspace = "session_data_on_fire_fabric_spark"

fabric.model_memory_analyzer(dataset = __xlmodel, workspace = __workspace)



## 8. Close the Loop — Write Back to the Lakehouse
Read from the semantic model, enrich it in Python, write it back.
Direct Lake picks up the new table immediately — no refresh needed.
**Power BI → Python → Power BI.**



### Read Sales from the semantic model


In [ ]:

# Same data Power BI is reading — now in a Spark DataFrame
df_sales_pd = fabric.read_table(dataset_name, "Sales")
df_sales = spark.createDataFrame(df_sales_pd)

display(df_sales.limit(5))



### Enrich — compute Sales Amount, add a Sales Tier


In [ ]:

from pyspark.sql.functions import col, when, round as spark_round

df_enriched = (
    df_sales
    .withColumn(
        "Sales Amount",
        spark_round(col("Quantity") * col("Unit Price"), 2)
    )
    .withColumn(
        "Sales Tier",
        when(col("Sales Amount") >= 1000, "High")
        .when(col("Sales Amount") >= 500,  "Medium")
        .otherwise("Low")
    )
    .filter(col("Quantity") > 0)
)

display(
    df_enriched
    .select("Order Number", "Order Date", "Quantity", "Unit Price", "Sales Amount", "Sales Tier")
    .limit(10)
)



### Write back — Direct Lake sees it immediately
Delta tables don't allow spaces in column names — rename them before writing.


In [ ]:

from pyspark.sql.functions import col

df_enriched_clean = df_enriched
for col_name in df_enriched.columns:
    df_enriched_clean = df_enriched_clean.withColumnRenamed(col_name, col_name.replace(" ", "_"))

(
    df_enriched_clean
    .write
    .mode("overwrite")
    .saveAsTable("FactSalesEnriched")
)

print(f"Done — {df_enriched_clean.count():,} rows written to FactSalesEnriched")
print("Direct Lake sees this immediately — no refresh needed.")
